# Semana 08
## Storytelling tecnico y anotacion de hallazgos

**Objetivo**: transformar un hallazgo exploratorio en una pieza explicativa con evidencia, anotaciones y recomendación.

**Herramientas teoricas de la semana**
- arquitectura narrativa
- titulos analiticos
- anotaciones
- evidencia vs recomendacion


### Agenda sugerida de 4 horas
- 0:00 - 0:30: estructura narrativa para datos
- 0:30 - 1:15: seleccion del hallazgo principal
- 1:15 - 2:10: construccion de grafico anotado
- 2:10 - 3:20: narrativa y tabla de soporte
- 3:20 - 4:00: presentacion corta y retroalimentacion


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-08"
OUTPUT_DIR = ensure_output_dir(WEEK)

clean, _ = clean_sales_data(introduce_quality_issues(make_base_sales(n=2200, seed=19), seed=19))
monthly_region = (
    clean.assign(month=clean['order_date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['month', 'region'], as_index=False)['sales']
    .sum()
)
monthly_region.head()


### Actividad principal
Escoge una región o categoría con un patrón interesante y conviértelo en una visualización explicativa.


In [ ]:
focus_region = 'South'
region_ts = monthly_region.query('region == @focus_region').copy()
peak_idx = region_ts['sales'].idxmax()
peak_month = region_ts.loc[peak_idx, 'month']
peak_value = region_ts.loc[peak_idx, 'sales']

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(region_ts['month'], region_ts['sales'], color='#111827', linewidth=2.8)
ax.scatter([peak_month], [peak_value], color='#9ca3af', s=120)
ax.annotate(
    f'Pico de ventas en {peak_month:%b %Y}
{peak_value:,.0f}',
    xy=(peak_month, peak_value),
    xytext=(peak_month, peak_value * 1.08),
    arrowprops=dict(arrowstyle='->', color='#374151'),
    fontsize=11,
)
ax.set_title('La region South alcanza su maximo de ventas a mitad de 2024')
ax.set_ylabel('sales')
plt.tight_layout()


In [ ]:
story_table = pd.DataFrame([
    {'stage': 'Contexto', 'content': 'Se analiza la evolucion mensual de ventas por region durante 2023-2024.'},
    {'stage': 'Pregunta', 'content': 'Que region muestra el cambio mas marcado y en que momento ocurre?'},
    {'stage': 'Evidencia', 'content': 'South registra un pico claro a mitad de 2024.'},
    {'stage': 'Insight', 'content': 'La region South concentra una aceleracion temporal mas fuerte que el resto.'},
    {'stage': 'Recomendacion', 'content': 'Investigar drivers comerciales o estacionales alrededor del pico detectado.'},
])
story_table


In [ ]:
save_for_tableau(monthly_region, WEEK, 'monthly_region')
save_for_tableau(story_table, WEEK, 'story_table')


### Uso teorico de herramientas
- `annotate` permite convertir un gráfico analítico en un gráfico explicativo.
- La `story_table` traduce directamente la arquitectura narrativa vista en clase.
- El CSV exportado puede usarse para una Tableau Story o para una lámina de presentación.
